# Aria Gen 2 HLOC: Two Mapping Recordings and an Independent Third Query

Map **Pilot `play_0` + `play_2`**, then query and replay **`play_1`**. The third recording never enters the map, features or retrieval index. Excluding keyframes from one mapping VRS is not independent recording validation.

Create the pinned environment, recursively initialize HLOC, install it with `--no-deps`, and apply `patches/hloc-workers.patch`; follow the README's macOS OpenMP note if needed. Obtain all three recordings through the official Pilot Explorer access/consent flow and download VRS, trajectory and calibration with `aria_dataset_downloader`. Use the original public Pilot MPS outputs in their shared graph, not a private joint-MPS rerun. This notebook does not download recordings, accept terms or start MPS requests on your behalf. Budget space for all three inputs and generated maps/outputs; no glasses are needed.

The workflow uses one **RGB-built map from two recordings**. Optional four-SLAM-camera queries use gPnP against that same map. Upstream components download model weights on first inference; weights are not bundled. Cells are intentionally unexecuted in source, with no fabricated outputs.


In [ ]:
import json
from pathlib import Path
from build_map import build_map
from check_inputs import check_inputs
from map_utils import file_sha256, Trajectory, write_json
from query_map import load_map, query_map
from replay_demo import replay

PILOT = Path('data/pilot')
MAP_VRS = [PILOT / 'play_0/video.vrs', PILOT / 'play_2/video.vrs']
MAP_MPS = [PILOT / 'play_0/mps', PILOT / 'play_2/mps']
TEST_VRS = PILOT / 'play_1/video.vrs'
TEST_MPS = PILOT / 'play_1/mps'
RGB_MAP = Path('maps/pilot_play02_rgb')
RGB_CACHE = Path('queries/pilot_play1_multi_rgb.json')

INPUTS = list(zip(MAP_VRS, MAP_MPS)) + [(TEST_VRS, TEST_MPS)]
for vrs, mps in INPUTS:
    required = (vrs, mps / 'slam/closed_loop_trajectory.csv', mps / 'slam/online_calibration.jsonl')
    if any(not path.is_file() for path in required):
        raise FileNotFoundError('Download all three Pilot play_0/play_2/play_1 VRS and MPS inputs using the README first')

MAP_HASHES = [file_sha256(vrs) for vrs in MAP_VRS]
TEST_HASH = file_sha256(TEST_VRS)
assert MAP_HASHES[0] != MAP_HASHES[1], 'The two mapping inputs must be different recordings'
assert TEST_HASH not in MAP_HASHES, 'The third query recording must not be either mapping input'
GRAPH_UIDS = [Trajectory(mps / 'slam/closed_loop_trajectory.csv').graph_uid for _, mps in INPUTS]
assert all(graph == GRAPH_UIDS[0] for graph in GRAPH_UIDS), 'Use the original public Pilot MPS outputs in one shared graph'
for vrs, mps in INPUTS:
    print(vrs.parent.name, check_inputs(vrs, mps))


## 1. Build only the two mapping recordings

The mapper rectifies motion-separated RGB keyframes and triangulates matches with known MPS poses. `max_keyframes=0` retains all selected frames. VRS/MPS lists are parallel: namespace `s000` is play_0 and `s001` is play_2; their clocks and calibration remain separate. Run this cell once with a new output directory. If you already built the exact play_0+play_2 RGB map using the README, skip this cell and proceed to fingerprint validation. There is no automatic rebuild. Use the exact two-input map, not a full-reference map containing play_1.


In [ ]:
build_stats = build_map(MAP_VRS, MAP_MPS, RGB_MAP, camera_set='rgb', max_keyframes=0)
print(json.dumps(build_stats, indent=2))


## 2. Verify exactly two ordered mapping fingerprints

Content fingerprints, not producer paths, verify membership. The ordered list must contain exactly play_0, then play_2, with no play_1 digest. Legacy maps without fingerprints cannot establish independence and are rejected by this notebook. Also verify the RGB configuration and shared MPS graph before querying an existing map.


In [ ]:
metadata = load_map(RGB_MAP)
mapping_hashes = metadata.get('source_vrs_sha256')
assert mapping_hashes == MAP_HASHES, 'Use the two-input RGB map in play_0, play_2 namespace order'
assert TEST_HASH not in mapping_hashes, 'Testing recording must not be in the map'
assert metadata['camera_set'] == 'rgb'
assert metadata['graph_uid'] == GRAPH_UIDS[0]


## 3. Query the independent third recording with RGB

Sample play_1 at stride 34, requesting 100 RGB queries. Query calibration comes from play_1. Query MPS poses provide optional same-graph reference comparisons, never retrieval or localization priors. Shared Multi-SLAM can jointly optimize the query poses; this is not independent physical ground truth. Keep failed and inaccurate accepted poses in the results.

One resident map/model initialization serves the batch. Initialization, first-query and warm-query timings differ; inspect the actual stage timings. Query images/features/matches stay in memory, with a byte-bounded reference-feature CPU cache. Neither map geometry nor acceptance thresholds are retuned.


In [ ]:
if RGB_CACHE.exists():
    raise FileExistsError('Choose a new query cache path instead of overwriting evidence')
rgb_result = query_map(RGB_MAP, TEST_VRS, TEST_MPS, mode='rgb', num_queries=100, stride=34)
assert rgb_result['source_identity_check'] == 'sha256'
assert rgb_result['query_in_mapping_recordings'] is False
assert rgb_result['query_vrs_sha256'] == TEST_HASH
assert rgb_result['same_mps_graph'] is True
print(json.dumps({key: value for key, value in rgb_result.items() if key != 'queries'}, indent=2))
RGB_CACHE.parent.mkdir(parents=True, exist_ok=True)
write_json(RGB_CACHE, rgb_result)


## 4. Optional four-camera gPnP queries

Use the same play_0+play_2 RGB map and play_1 query recording. A rig frameset spans at most 2 ms in corrected capture time. The stride-208 example requests 50 queries across approximately the full recording. The measured two-recording-map SLAM supplement used endpoint-inclusive uniformly spaced indices (208/209 gaps), not precisely this stride schedule. Report this run's actual counts and results rather than assuming the measured SLAM score or batch timing applies to it.


In [ ]:
slam_result = query_map(RGB_MAP, TEST_VRS, TEST_MPS, mode='slam', num_queries=50, stride=208)
assert slam_result['source_identity_check'] == 'sha256'
assert slam_result['query_in_mapping_recordings'] is False
assert slam_result['query_vrs_sha256'] == TEST_HASH
assert slam_result['same_mps_graph'] is True
print(json.dumps({key: value for key, value in slam_result.items() if key != 'queries'}, indent=2))


## 5. Replay only the third recording's VIO in Rerun

Use **play_1 VIO and its RGB query cache** against the play_0+play_2 map. Replay simulates a latest-frame worker using cached measured query delays; it does not rerun neural models, interpolate MPS poses into the output or validate physical USB streaming. JSONL retains every valid mapped VIO sample after alignment; pose rendering is capped at 30 Hz. Accepted visual outliers can corrupt alignment.

The separately executed native replay CLI and RRD verification passed, as did the row-wise JSONL/PoseAlignment parity audit: **271,964 recorded VIO samples, 265,344 mapped JSONL poses, 6,620 initially unaligned/lost samples**. All 100 RGB cache entries were retained; the scheduler produced 56 in-coverage scheduled query arrivals and 55 accepted fixes, with 42 superseded entries, one capture outside VIO coverage and one late arrival. Failed anchor 37 was scheduled; failed anchor 75 was superseded by the latest-pending policy, not manually removed. See [recorded replay statistics](benchmarks/pilot_multitrajectory.json).

First mapped output was 8.242 seconds after the first recorded VIO sample, not process start/cold initialization; first-fix capture-to-arrival was separately 6.205 seconds. The 803.03 Hz mapped-pose rate is in recorded device time, not visual-fix rate or live USB throughput. RRD shows all sparse-map positions with uniform colors and a 640px/JPEG90 RGB preview, not original point RGB/full-resolution texture; the separate original-color point export is not bundled here.

Run `rerun --bind 127.0.0.1 outputs/pilot_play1_multi_replay/view.rrd` to inspect the map, pose, RGB and fix age. This notebook itself remains unexecuted with no outputs. Recorded replay evidence is not physical hardware validation, final asset readiness or public-release approval. `live_demo.py` is a separate SDK streaming CLI for a map of the actual environment, not for unrelated rooms.


In [ ]:
replay_stats = replay(RGB_MAP, TEST_VRS, TEST_MPS, RGB_CACHE, Path('outputs/pilot_play1_multi_replay'))
print(json.dumps(replay_stats, indent=2))


## Measured two-recording map / third-recording queries

The completed play_0+play_2 RGB map has **1,360 keyframes (666 + 694), 135,213 landmarks and 14,870 cross-recording tracks**. All 150 fixed play_1 queries completed, with source fingerprints and map contents checked. **RGB:** 98/100 gates, 99 reference-covered queries, 97 accepted with a reference, 89/99 within 0.10 m AND 5 degrees, 93/99 within 0.25 m AND 5 degrees. **Four SLAM:** 48/50 gates, 48 reference-covered queries, 47 accepted with a reference and 47/48 within both thresholds. Accuracy denominators include reference-covered failures. Maximum accepted reference disagreements were 0.418 m / 13.23 degrees RGB and 0.075 m / 2.29 degrees SLAM. No accepted reference-covered pose exceeded 1 m or 170 degrees in this run; gate acceptance is not accuracy or a false-positive guarantee. See [measurements and exact protocol](benchmarks/pilot_multitrajectory.json).

RGB used indices 0, 34, ..., 3366, matching the primary command/API sampling. The measured 50-query SLAM supplement used endpoint-inclusive uniform indices with 208/209 gaps and last index 10200, not the exact stride-208 schedule above. These frozen, previously exposed anchors are not a fresh blind test; shared original public MPS reference poses are not independent physical ground truth.

Observed CPU query median/p90 was 6.10/7.02 seconds RGB and 12.85/13.89 seconds SLAM. Preparation took 8.66 seconds and recording/trajectory opening 18.80 seconds, separately. First RGB query took 8.37 seconds; subsequent RGB median/p90 was 6.06/6.96 seconds. One prepared backend served RGB then SLAM, unlike separate CLI initialization. Concurrent eight-thread CPU jobs prevent a controlled speed-gain or cold-start comparison; peak RSS was about 4.09 GiB, not bounded by the 256 MiB feature-array cache. The source cells remain unexecuted with no outputs. The separately executed recorded replay passed native CLI, JSONL parity and RRD checks, as described above; this is not physical hardware qualification, complete release-asset readiness or public-release approval.

No cross-day, unseen-location or real-time guarantee is made. Keep raw VRS/MPS, maps/features, weights, query caches, local replay outputs and executed notebook outputs out of release source. Separately reviewed full-reference assets, including any new joint-MPS rerun, do not replace this public independent-query protocol and are not bundled or hosted by this repository.
